In [9]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
from langchain_core.documents import Document

In [3]:
sample_doc = Document(
    page_content="This is a sample document",
    metadata={"source": "sample.pdf"}
)

In [10]:
sample_doc

In [8]:
type(sample_doc)

In [6]:
from langchain_community.document_loaders.text import TextLoader
loader = TextLoader("/content/Python.txt", encoding = "utf-8")

In [7]:
document = loader.load()
display(document)

[Document(metadata={'source': '/content/Python.txt'}, page_content='\ufeffPython is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automati

In [8]:
# from langchain_community.document_loaders.pdf import PyPDFLoader # for simple pdfs
# pdf_loader = PyPDFLoader("/content/research2.pdf")

In [7]:
from langchain_community.document_loaders.pdf import PyMuPDFLoader
pdf_loader = PyMuPDFLoader("/content/research2.pdf")
document1 = pdf_loader.load()
document1

Ingestion pipeline -


In [10]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [11]:
def load_all_pdfs():
  folder_path = "/content/"
  num_docs = 0
  all_docs = []

  for filename in os.listdir(folder_path):
    if filename.lower().endswith(".pdf"):
      pdf_path = os.path.join(folder_path, filename)

      loader = PyPDFLoader(pdf_path)
      doc = loader.load()

      all_docs.extend(doc)
      num_docs += 1

  print("total pdf: ", num_docs)
  print("total pages: ", len(all_docs))

  return all_docs

In [12]:
all_pdf_documents = load_all_pdfs()

total pdf:  1
total pages:  21


create chuncks of data

In [6]:
!pip install langchain_text_splitters

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(document , chunk_size = 500 , chunk_overlap = 50 ):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(document)
    return chunked_docs

In [15]:
chunks = split_docs(all_pdf_documents)

In [16]:
len(chunks)

244

In [17]:
# chunks

Embedding


In [18]:
from sentence_transformers import SentenceTransformer

In [19]:
class EmbeddingManager:


  def __init__(self , model_name = "all-MiniLM-L6-v2"): # all-MiniLM-L6-v2 sentence transformer model from HF

    self.model_name = (model_name)
    print("loading model.." , self.model_name)

    self.model = SentenceTransformer(self.model_name)
    print("embedding dimensions=" , self.model.get_sentence_embedding_dimension())




  def generate_embeddings(self , text):
    embeddings =  self.model.encode(text , show_progress_bar = True)

    print("embedding shape" , embeddings.shape)
    return embeddings

In [5]:
embedding_manager = EmbeddingManager()

vector store  --->  query -> query embedding -> vectordb



In [21]:
import chromadb
import uuid  # create indexes for values

In [22]:
# class VectorStoreManager:
#   def __init__(self ,persist_directory = "/content/vector_store", collection_name = "pdf_documents"):
#     self.persist_directory = persist_directory
#     self.collection_name = collection_name.strip()
#     self.collection = None
#     self.client = None

#     # Initialize the store immediately on creation
#     self._initialize_store()

#   def _initialize_store(self):
#     os.makedirs(self.persist_directory, exist_ok=True)

#     self.client = chromadb.PersistentClient(path=self.persist_directory)

#     self.collection = self.client.get_or_create_collection(
#         name=self.collection_name,
#         metadata={"description": "vector store collection for pdf embeddings in RAG"}
#       )

#     print("Initialized vector store collection: " , self.collection_name)
#     print("Current documents in collection:", self.collection.count())



# def add_documents(self , documents , embeddings):
#   if len(documents) != len(embeddings):
#     raise ValueError("Number of documents and embeddings does not be the same.")


# # store ---> ids , embeddings , doc , metadata
#     id=[]
#     all_metadata =[]
#     document_content = []
#     embeddings_list = []


#     for i, (doc ,embedding) in enumerate(zip(documents , embeddings)):
#        doc_id = f"doc_123_{uuid.uuid4()}"

#        metadata = dict(doc.metadata)
#        metadata["doc_index"] = i
#        metadata["content_length"] = len(doc)
#        all_metadata.append(metadata)


#        document_content.append(doc.page_content)
#        embeddings_list.append(embedding.tolist())



#        self.collection.add(
#            ids=ids,
#            metadatas=all_metadata,
#            documents=document_content,
#            embeddings=embeddings_list
#        )
# print("total documents added in collection: " , len(documents_content))
# print("docs in collection:", self.collection.count())


class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

In [23]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [4]:
uuid.uuid4()

In [3]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Rag Retrieval pipeline --->   vector db -> context + Query -> llm -> o/p


In [26]:
from sklearn.metrics.pairwise import cosine_similarity

In [27]:
class RAGretriever:
  def __init__(self , embedding_manager , vector_store):
    self.vector_store = vector_store
    self.embedding_manager = embedding_manager

  def retrieve(self , query , top_k=5 , score_threshold=0.0):
    query_embedding = self.embedding_manager.generate_embeddings([query])[0]# text is list thus query passed


#semnatic search
    results = self.vector_store.collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k,
    )



    #cosine similarity

    retrieved_docs=[]
    if results["documents"] and results["documents"][0]:
      ids = results["ids"][0]
      metadatas = results["metadatas"][0]
      documents = results["documents"][0]
      distances = results["distances"][0]


      for i , (doc_id , metadata , doc , distance) in enumerate(zip(ids , metadatas , documents , distances)):
        similarity_score = 1 - distance
        if similarity_score >= score_threshold:
          retrieved_docs.append({
              "id": doc_id,
              "metadata": metadata,
              "document": doc, # Changed from 'document' to 'doc' to use the iteration variable
              "similarity_score": similarity_score,
              "rank" : i+1
       })
      print(f"retrieved {len(retrieved_docs)} documents")

    else:
      print("no documents found")

    return retrieved_docs

In [28]:
rag_retriever = RAGretriever(embedding_manager, vector_store)

In [29]:
rag_retriever.retrieve("What is RAG")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape (1, 384)
retrieved 5 documents


[{'id': 'doc_eb5c3da9-7775-40bd-97ab-bd40e4c0fb4d',
  'metadata': {'page': 0,
   'keywords': '',
   'author': '',
   'creator': 'LaTeX with hyperref',
   'source': '/content/research2.pdf',
   'producer': 'pdfTeX-1.40.25',
   'total_pages': 21,
   'creationdate': '2024-03-28T00:54:45+00:00',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'page_label': '1',
   'trapped': '/False',
   'subject': '',
   'content_length': 288,
   'moddate': '2024-03-28T00:54:45+00:00',
   'title': '',
   'doc_index': 11},
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'similarity_score': 0.5773242115974426,
  'rank': 1},
 {'id': 'doc_122c7f15-8422-4eb3-b041-94e13efe628d',
  'met

integerate with LLM - GROQ

In [ ]:
API_KEY_GROQ = "YOUR_OWN_API_KEY"

In [2]:
!pip install langchain-groq

In [32]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=API_KEY_GROQ,
    model="qwen/qwen3-32b",
    temperature=0.1,
    max_tokens=1024,#can be none
)

In [33]:
# Generate RAG output

def generate_output(query , rag_retriever , llm , top_k=3):
  results = rag_retriever.retrieve(query, top_k)

  # Corrected syntax and variable names
  context = "\n".join([doc["document"] for doc in results]) if results else ""

  if not context:
      print("No relevant documents found")

  prompt = [
      {
          "role": "system",
          "content": "You are given context to generate the answer for the query."
      },
      {
          "role": "user",
          "content": f"""
Context:
{context}

Query:
{query}
"""
      }
  ]

  response = llm.invoke(prompt) # expect a list
  return response.content

In [1]:
answer = gnerate_output("what is RAG" , rag_retriever ,)

In [36]:
print(answer)

<think>
Okay, the user is asking "what is RAG". Let me start by recalling the context provided. The context mentions a survey paper about RAG, which stands for Retrieval-Augmented Generation. The paper reviews state-of-the-art RAG methods, their evolution through paradigms like naive RAG, and effective frameworks. It also talks about evaluation methods, tasks, datasets, and future directions.

So, RAG is a method that combines retrieval and generation. The user probably wants a concise definition. I should explain that RAG uses a retrieval component to fetch relevant information from a corpus and then uses a generator (like a language model) to produce answers based on that retrieved info. The context also mentions its integration with LLMs, so maybe mention that it enhances LLMs by providing external knowledge.

I need to make sure to highlight the key components: retrieval and generation. Also, note that it's used to improve the accuracy and relevance of responses by leveraging exter